<a href="https://colab.research.google.com/github/Awwanna/geoinformatika/blob/main/CC_in_GEE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

import and installing libraries

In [1]:
import ee                           # Earth Engine
import geemap                       # geemap library
import matplotlib.pyplot as plt     # for figure visualisation
import pandas as pd                 # for data storage and manipulation
import numpy as np                  # array handling
#import ruptures as rpt              # for breakpoint detection


authentication and initialisation

In [2]:
ee.Authenticate()

In [3]:
ee.Initialize(project = 'ee-wildovaan')
print('Welcome to the Earth Engine Python API!')

Welcome to the Earth Engine Python API!


# Study area - Podgorica, Montenegro
- start and end date of wildfire: 20 Mar 2022 - 28 Mar 2022
- burnt area (ha): 1074
- one of the 130 wildfires in Montenegro in 2022
- main burnt ecosystems:
    - forest broadleaves 61.2 %,
    - transitional woodland-shurbland 36.2 %
    - agricultural areas 2.5 %

Define the study area

In [4]:
study_area = ee.Geometry.Polygon(
        [[[19.344864, 42.812025],
          [19.481506, 42.812025],
          [19.481506, 42.729866],
          [19.344864, 42.729866]]],
        )
Map = geemap.Map()
Map.addLayer(study_area)
Map.centerObject(study_area, 10)
display(Map)

Map(center=[42.77095672333046, 19.413185000001324], controls=(WidgetControl(options=['position', 'transparent_…

In [5]:
# inspect the study_area
study_area

ee.Geometry({
  "functionInvocationValue": {
    "functionName": "GeometryConstructors.Polygon",
    "arguments": {
      "coordinates": {
        "constantValue": [
          [
            [
              19.344864,
              42.812025
            ],
            [
              19.481506,
              42.812025
            ],
            [
              19.481506,
              42.729866
            ],
            [
              19.344864,
              42.729866
            ]
          ]
        ]
      },
      "evenOdd": {
        "constantValue": true
      }
    }
  }
})

In [6]:
start_date = '2022-03-01'
end_date = '2022-05-01'
# Filter the Sentinel-2 image collection
dataset = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')# Import data
      .filterBounds(study_area)                             # Spatial filter
      .filterDate('2022-03-01', '2022-05-01')               # Temporal filter
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30))  # Metadata filter
      .sort('CLOUDY_PIXEL_PERCENTAGE'))                     # Sort data

# Print data to the Console
display(dataset)
display(dataset.first()) # Display image with lowest cloudy pixel percentage

In [7]:
# Add True color RGB composite to the map
Map.addLayer(
    dataset.first(),
    {'min': 0, 'max': 3000, 'bands': 'B4,B3,B2'},
    "S2 RGB Image"
)

# Add a false color composite
Map.addLayer(
    dataset.first(),
    {'min': 0, 'max': 3000, 'bands': 'B8,B4,B3'},
    "S2 NIR-R-G Image"
)

Map.centerObject(study_area)
display(Map)
Map.remove_last_drawn()

Map(center=[42.77095672333046, 19.413185000001324], controls=(WidgetControl(options=['position', 'transparent_…

#Comparing of sentinel RGB images of the study area before and after the wildfire

In [8]:
# Define the burned area
burned_area = ee.Geometry.Polygon(
        [[[19.344864, 42.812025],
          [19.481506, 42.812025],
          [19.481506, 42.729866],
          [19.344864, 42.729866]]],
        )
style = {'color': 'eeb61bff', 'width': 3, 'lineType': 'solid', 'fillColor': '00000019'}

# Filter sentinel data - year before the wildfire
dataset_before_wildfire = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterBounds(burned_area)
      .filterDate('2021-03-29', '2021-04-30')
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30)) # filter the images with the clouds less than 30%
      .sort('CLOUDY_PIXEL_PERCENTAGE')) # the most cloudless picture is the first in the dataset

# Add the first picture to map
Map1 = geemap.Map() # map for the before/after wildfire
Map1.addLayer(
    dataset_before_wildfire.first(),
    {'min': 0, 'max': 3000, 'bands': 'B4,B3,B2'},
    "S2 RGB Image before wildfire"
)

Map1.centerObject(burned_area,10)
display(Map1)

Map(center=[42.77095672333046, 19.413185000001324], controls=(WidgetControl(options=['position', 'transparent_…

# Cloud masking using CloudScore+

In [9]:
# Load the CloudScore+ dataset containing cloud probabilities for each image
csPlus = ee.ImageCollection('GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED')

# The threshold for masking; values between 0.50 and 0.65 generally work well.
CLEAR_THRESHOLD = 0.60
QA_BAND = 'cs'

# Function to mask clouds using the Sentinel-2 QA band
def maskS2clouds(img):
  mask = img.select(QA_BAND).gte(CLEAR_THRESHOLD)
  return img.updateMask(mask)

In [10]:
# Filter the Sentinel-2 image collection
S2_montenegro = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')     # Import data
      .filterBounds(burned_area)                            # Spatial filter
      .filterDate('2021-03-29', '2021-04-30')               # Temporal filter
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30))  # Metadata filter
      .linkCollection(csPlus,[QA_BAND])                     # Link with CloudScore+
      .sort('CLOUDY_PIXEL_PERCENTAGE')
      .map(maskS2clouds)
      )
S2_cloudless = S2_montenegro.map(maskS2clouds)                         # Mask out clouds
display(S2_cloudless)
# Create a median composite and clip it
#cloudfree_montenegro = S2_montenegro.median().clip(burned_area)
# Print the size of the dataset used for composite creation
#print('The size of the entire image collection for 2022 (so far) is', S2_montenegro.size().getInfo())

# Filter sentinel data - before the wildfire
dataset_before_wildfire_cs = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterBounds(burned_area)
      .filterDate('2021-03-05', '2021-03-19')
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30)) # filter the images with the clouds less than 30%
      .linkCollection(csPlus,[QA_BAND])
      .sort('CLOUDY_PIXEL_PERCENTAGE'))

# Filter sentinel data - after the wildfire
dataset_after_wildfire_cs = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterBounds(burned_area)
      .filterDate('2022-03-29', '2022-04-30')
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30)) # filter the images with the clouds less than 30%
      .linkCollection(csPlus,[QA_BAND])
      .sort('CLOUDY_PIXEL_PERCENTAGE')) # the most cloudless picture is the first in the dataset
# Add the unmasked
Map2 = geemap.Map()

# Add the unmasked picture after the wildfire
Map2.addLayer(
    dataset_before_wildfire_cs.first(),
    {'min': 0, 'max': 3000, 'bands': 'B4,B3,B2'},
    "S2 RGB Image before wildfire, CloudScore+"
)
Map2.addLayer(
    dataset_after_wildfire_cs.first(),
    {'min': 0, 'max': 3000, 'bands': 'B4,B3,B2'},
    "S2 RGB Image after wildfire, CloudScore+"
)

Map2.addLayer(burned_area, vis_params=style)
Map2.centerObject(burned_area,10)

In [11]:
display(Map2)

Map(center=[42.77095672333046, 19.413185000001324], controls=(WidgetControl(options=['position', 'transparent_…

# Calculate the NBR
Couldn't connect the nbr to the Bands

In [12]:
# Function for adding NBR index
def addNBR(image):
    nbr = image.normalizedDifference(['B8', 'B12']).rename('NBR')
    return image.addBands(nbr)

# Add NBR to each image - before wildfire
before_wildfire_nbr = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterBounds(burned_area)
      .filterDate('2021-03-05', '2021-03-19')
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30)) # filter the images with the clouds less than 30%
      .linkCollection(csPlus,[QA_BAND])
      .map(addNBR)
      .first())
# Add NBR to each image - after wildfire
after_wildfire_nbr = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterBounds(burned_area)
      .filterDate('2022-03-29', '2022-04-30')
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30)) # filter the images with the clouds less than 30%
      .linkCollection(csPlus,[QA_BAND])
      .map(addNBR)
      .first())

# Print information about bands
display(before_wildfire_nbr)
display(after_wildfire_nbr)

## Add the NBR to Map3

In [19]:
Map3 = geemap.Map()

# Calculate NBR for the first image in the before wildfire collection
nbr_before = before_wildfire_nbr.select('NBR')
nbr_after = after_wildfire_nbr.select('NBR')

Map3.addLayer(
    nbr_before,
    {'min': -1, 'max': 1, 'palette': ['#d7191c', '#fdae61', '#ffffbf', '#a6d96a', '#1a9641']},
    'NBR Image Before Wildfire'
)

Map3.addLayer(
    nbr_after,
    {'min': -1, 'max': 1, 'palette': ['#d7191c', '#fdae61', '#ffffbf', '#a6d96a', '#1a9641']},
    'NBR Image After Wildfire'
)
legend_keys = ["Highest normalized burn ratio lost", "Lowest normalized burn ratio"]
legend_colors = ['#d7191c','#1a9641']

Map3.add_legend(keys=legend_keys, colors=legend_colors, position="bottomleft")


Map3.addLayer(burned_area, vis_params=style)
Map3.centerObject(burned_area, 12)
display(Map3)

Map(center=[42.77095672333046, 19.413185000001324], controls=(WidgetControl(options=['position', 'transparent_…